# **직접 만든 데이터로 파인튜닝**

# 1.환경준비

## (1) 라이브러리 설치

In [ ]:
!pip install ultralytics roboflow -q

## (2) 라이브러리 불러오기

In [1]:
from ultralytics import settings, YOLO
import matplotlib.pyplot as plt
import cv2
import os
from IPython.display import Video

from roboflow import Roboflow

* 폴더 내 이미지 개수 확인

In [2]:
def image_count(path) :
    image_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]  # YOLO에서 지원하는 이미지 확장자

    # valid 폴더에서 이미지 파일 개수 확인
    image_count = len([f for f in os.listdir(path) if f.lower().endswith(tuple(image_extensions))])

    return image_count

## (3) YOLO 설정

* 파일 경로 설정

In [3]:
# 콜랩 파일 탭에 보이는 경로('/content/')로 변경해 봅시다.
settings['datasets_dir'] = './'
settings.update()
settings

{'settings_version': '0.0.6',
 'datasets_dir': './',
 'weights_dir': 'weights',
 'runs_dir': 'runs',
 'uuid': 'e93724c3a62b829fe2022931d0c4e07c6e9e5bbc21bb08e52bb2b401c0a189fa',
 'sync': True,
 'api_key': '',
 'openai_api_key': '',
 'clearml': True,
 'comet': True,
 'dvc': True,
 'hub': True,
 'mlflow': True,
 'neptune': True,
 'raytune': True,
 'tensorboard': False,
 'wandb': False,
 'vscode_msg': True,
 'openvino_msg': True}

# 2.모델링

## (1) 데이터셋 다운로드

In [ ]:
rf = Roboflow(api_key="")
project = rf.workspace("computer-vision-ys9wb").project("aivle-yolo-yfmhy")
version = project.version(4)
dataset = version.download("yolov11")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to AIVLE-yolo-4 in yolov11:: 100%|██████████| 125/125 [00:00<00:00, 1864.37it/s]


* 이미지 개수 확인

In [ ]:
projcet_name = "aivle-yolo-4"
# train 이미지
cnt = image_count('./' + projcet_name+ '/train/images')
print('* train 이미지 수 :', cnt)

# valid 이미지
cnt = image_count('./' + projcet_name+ '/valid/images')
print('* valid 이미지 수 :', cnt)

* train 이미지 수 : 341
* valid 이미지 수 : 33


## (2) 모델 다운로드

- 모델의 구조와 해당 구조에 맞게 사전 학습된 가중치를 불러온다.
- Parameters
    * model : 모델 구조 또는 모델 구조 + 가중치 설정. task와 맞는 모델을 선택해야 한다.
    * task : detect, segment, classify, pose 중 택일

In [13]:
model = YOLO(model='yolo11n.pt', task='detect')

## (3) 파인튜닝

* 모델 학습
    * 파라미터 설명 : [Parameters](https://docs.ultralytics.com/modes/train/#train-settings)

In [15]:
results_train = model.train(
    data='./' + projcet_name+ '/data.yaml',
    epochs=20,
    patience=5,
    optimizer='Adam',
)

Ultralytics 8.3.225  Python-3.10.19 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 3070 Ti, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./futbol-players-9/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam, overlap_mask=True, patience=5, perspective=0.0, plots=True

## (4) 평가

In [16]:
results_val = model.val(
    data='./' + projcet_name+ '/data.yaml',
    split='val',
    imgsz=640,
    batch=16,
    conf=0.25,
    iou=0.5
)
print("\n[평가 결과 요약]")
print(results_val.results_dict)


Ultralytics 8.3.225  Python-3.10.19 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 3070 Ti, 8192MiB)
YOLO11n summary (fused): 100 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 94.019.8 MB/s, size: 19.9 KB)
val: Scanning C:\Users\USER\Documents\GitHub\AIVLE_School\(7주차) 시각 지능과 멀티모달 AI\futbol-players-9\valid\labels.cache... 33 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 33/33  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 0.7it/s 4.5s1.5ss
                   all         33        407      0.954      0.808      0.901      0.541
                     0         33        359      0.983      0.965      0.988      0.645
                     1         20         20      0.938        0.9      0.942      0.682
                     2         28         28       0.94       0.56      0.773      0.296
Speed: 11.7ms preprocess, 15.1ms inference, 0.0ms loss, 1.4ms postprocess p

## (5) 예측해보기

In [17]:
results_pred = model.predict(
    source='./' + projcet_name+ '/valid/images',  # 검증 이미지 폴더
    imgsz=640,
    conf=0.25,
    save=True,             # 예측 이미지 저장
    save_txt=True,         # 예측 bounding box 저장
    project='runs/detect', # 결과 폴더
    name='predict_dogcatbird',
    show=False
)


image 1/33 c:\Users\USER\Documents\GitHub\AIVLE_School\(7)    AI\futbol-players-9\valid\images\1-fps-2_00007_jpeg_jpg.rf.8c63e7f01364fa1b94be7ba04560e832.jpg: 640x640 10 0s, 1 1, 1 2, 15.1ms
image 2/33 c:\Users\USER\Documents\GitHub\AIVLE_School\(7)    AI\futbol-players-9\valid\images\1-fps-2_00013_jpeg_jpg.rf.fef00c16e7b285e846dd5051474e57bb.jpg: 640x640 5 0s, 2 2s, 13.8ms
image 3/33 c:\Users\USER\Documents\GitHub\AIVLE_School\(7)    AI\futbol-players-9\valid\images\1-fps-2_00015_jpeg_jpg.rf.0f35004f16be298e9acf40ff4670f72a.jpg: 640x640 5 0s, 1 2, 15.5ms
image 4/33 c:\Users\USER\Documents\GitHub\AIVLE_School\(7)    AI\futbol-players-9\valid\images\1-fps-2_00020_jpeg_jpg.rf.d88ad6cdc1525d268bd42291eb40e212.jpg: 640x640 11 0s, 1 1, 1 2, 16.6ms
image 5/33 c:\Users\USER\Documents\GitHub\AIVLE_School\(7)    AI\futbol-players-9\valid\images\1-fps-2_00023_jpeg_jpg.rf.508cb938977b8d5dfbce391d3e4953fe.jpg: 640x640 7 0s, 1 2, 16.0ms
image 6/33 c:\Users\USER\Documents\GitHub\AIVLE_School\(7)    

In [ ]:
results_pred[0].show()  